# GTEx Tissue-Level Expression

The NIH Common Fund Genotype-Tissue Expression, or GTEx, program created a resource for studying gene expression across human tissues. Compare candidate-gene expression in heart atrial appendage and left ventricle using a saved GTEx v10 response.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests

DATA_DIR = Path("data") if Path("data").exists() else Path("../data")
gtex = pd.read_csv(DATA_DIR / "gtex_expression.csv")
gtex.head()

## Optional live API request

The analysis uses the saved file by default. Run the next function only when you want to compare it with the current API.

In [ ]:
GTEX_EXPRESSION_URL = (
    "https://gtexportal.org/api/v2/expression/medianGeneExpression"
)


def fetch_gtex_expression(
    gencode_ids: list[str],
    tissue_ids: list[str],
    timeout_seconds: int = 30,
) -> pd.DataFrame:
    """Return current GTEx median expression for explicit genes and tissues."""
    response = requests.get(
        GTEX_EXPRESSION_URL,
        params={
            "gencodeId": gencode_ids,
            "datasetId": "gtex_v10",
            "tissueSiteDetailId": tissue_ids,
        },
        timeout=timeout_seconds,
    )
    response.raise_for_status()
    return pd.DataFrame(response.json()["data"])


# Example, intentionally not executed during routine notebook runs:
# live_gtex = fetch_gtex_expression(
#     gtex["gencode_id"].unique().tolist(),
#     ["Heart_Atrial_Appendage", "Heart_Left_Ventricle"],
# )

In [ ]:
heart_expression = (
    gtex.pivot(
        index="gene_symbol",
        columns="tissue_name",
        values="median_tpm",
    )
    .sort_values("Heart - Left Ventricle", ascending=False)
)
heart_expression

In [ ]:
axis = heart_expression.plot.bar(
    color=["#3d64b3", "#764c82"],
    figsize=(9, 5),
)
axis.set_ylabel("Median expression (TPM)")
axis.set_xlabel("Candidate gene")
axis.set_title("GTEx v10 heart-tissue expression")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Interpretation

All five genes have measurable median expression in both heart tissues in this dataset. The values add tissue context. They do not show what the listed variants do.